In [1]:
!pip install --quiet --upgrade google-cloud-bigquery google-cloud-storage requests pandas db-dtypes

import time, requests
import pandas as pd
from google.cloud import bigquery, storage

PROJECT_ID    = "qwiklabs-gcp-00-c521a9ba0b6e"
DATASET_ID    = "aero_alerts"
AIRPORTS_TBL  = "airports"
LARGE_TBL     = "large_us_airports"
FORECAST_TBL  = "airport_forecasts"
ALERTS_TBL    = "airport_alerts"

CONNECTION_ID = "gemini_conn"
MODEL_NAME    = "gemini_model"
REGION        = "US"

BUCKET_NAME   = f"{PROJECT_ID}-aero-alerts"

bq = bigquery.Client(project=PROJECT_ID)
print("Client ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 80.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Client ready.


In [2]:
storage_client = storage.Client(project=PROJECT_ID)

try:
    storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Created bucket {BUCKET_NAME}")
except Exception as e:
    print(f"Bucket may already exist: {e}")

!gsutil cp gs://labs.roitraining.com/data-to-ai-workshop/airports.csv /tmp/airports.csv
!gsutil cp /tmp/airports.csv gs://{BUCKET_NAME}/airports.csv
print("CSV staged in your bucket.")

Created bucket qwiklabs-gcp-00-c521a9ba0b6e-aero-alerts
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://labs.roitraining.com/data-to-ai-workshop/airports.csv...
/ [1 files][ 11.7 MiB/ 11.7 MiB]                                                
Operation completed over 1 objects/11.7 MiB.                                     
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///tmp/airports.csv [Content-Type=text/csv]...
- [1 files][ 11.7 MiB/ 11.7 MiB]                                                
Operation completed over 1 objects/11.7 MiB.        

In [3]:
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
ds.location = REGION
bq.create_dataset(ds, exists_ok=True)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
bq.load_table_from_uri(
    f"gs://{BUCKET_NAME}/airports.csv",
    f"{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TBL}",
    job_config=job_config,
).result()

tbl = bq.get_table(f"{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TBL}")
print(f"Loaded {tbl.num_rows:,} airports.")

Loaded 82,893 airports.


In [4]:
bq.query(f"""
    SELECT type, COUNT(*) AS n
    FROM `{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TBL}`
    WHERE iso_country = 'US'
    GROUP BY type
    ORDER BY n DESC
""").to_dataframe()

,type,n
0,small_airport,15225
1,heliport,8137
2,closed,7098
3,medium_airport,832
4,seaplane_base,650
5,large_airport,71
6,balloonport,29


In [5]:
bq.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{LARGE_TBL}` AS
SELECT
    ident,
    icao_code,
    iata_code,
    name,
    municipality,
    iso_region,
    latitude_deg,
    longitude_deg
FROM `{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TBL}`
WHERE type = 'large_airport'
  AND iso_country = 'US'
""").result()

large_df = bq.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{LARGE_TBL}`
""").to_dataframe()

print(f"{len(large_df)} large US airports.")
large_df.head()

71 large US airports.


,ident,icao_code,iata_code,name,municipality,iso_region,latitude_deg,longitude_deg
0,PANC,PANC,ANC,Ted Stevens Anchorage International Airport,Anchorage,US-AK,61.179004,-149.992561
1,KPHX,KPHX,PHX,Phoenix Sky Harbor International Airport,Phoenix,US-AZ,33.435302,-112.005905
2,KSFO,KSFO,SFO,San Francisco International Airport,San Francisco,US-CA,37.619806,-122.374821
3,KLAX,KLAX,LAX,Los Angeles International Airport,Los Angeles,US-CA,33.942501,-118.407997
4,KOAK,KOAK,OAK,San Francisco Bay Oakland International Airport,Oakland,US-CA,37.720085,-122.221184


In [7]:
HEADERS = {"User-Agent": "AeroAlerts-Workshop (simar.sharma@zionclouds.com)"}

def get_forecast(lat, lng):
    """Two-step NWS lookup: points -> forecast URL -> forecast periods."""
    try:
        pts = requests.get(
            f"https://api.weather.gov/points/{lat},{lng}",
            headers=HEADERS, timeout=15
        )
        if pts.status_code != 200:
            return None
        forecast_url = pts.json()["properties"]["forecast"]

        fc = requests.get(forecast_url, headers=HEADERS, timeout=15)
        if fc.status_code != 200:
            return None
        periods = fc.json()["properties"]["periods"]
        if not periods:
            return None

        p = periods[0]
        return {
            "period_name":     p.get("name"),
            "temperature":     p.get("temperature"),
            "temperature_unit": p.get("temperatureUnit"),
            "wind_speed":      p.get("windSpeed"),
            "wind_direction":  p.get("windDirection"),
            "short_forecast":  p.get("shortForecast"),
            "detailed_forecast": p.get("detailedForecast"),
        }
    except Exception as e:
        print(f"  error @ {lat},{lng}: {e}")
        return None

rows = []
for _, a in large_df.iterrows():
    f = get_forecast(a["latitude_deg"], a["longitude_deg"])
    if f:
        rows.append({
            "ident": a["ident"],
            "icao_code": a["icao_code"],
            "iata_code": a["iata_code"],
            "name": a["name"],
            "municipality": a["municipality"],
            "iso_region": a["iso_region"],
            "latitude_deg": a["latitude_deg"],
            "longitude_deg": a["longitude_deg"],
            **f,
        })
    time.sleep(0.5)
    print(f"  fetched {len(rows)}/{len(large_df)}", end="\r")

forecast_df = pd.DataFrame(rows)
print(f"\nGot forecasts for {len(forecast_df)} airports.")
forecast_df.head()

  fetched 71/71
Got forecasts for 71 airports.


,ident,icao_code,iata_code,name,municipality,iso_region,latitude_deg,longitude_deg,period_name,temperature,temperature_unit,wind_speed,wind_direction,short_forecast,detailed_forecast
0,PANC,PANC,ANC,Ted Stevens Anchorage International Airport,Anchorage,US-AK,61.179004,-149.992561,Overnight,48,F,5 mph,S,Partly Cloudy,"Partly cloudy, with a low around 48. South win..."
1,KPHX,KPHX,PHX,Phoenix Sky Harbor International Airport,Phoenix,US-AZ,33.435302,-112.005905,Overnight,77,F,5 mph,NE,Clear,"Clear, with a low around 77. Northeast wind ar..."
2,KSFO,KSFO,SFO,San Francisco International Airport,San Francisco,US-CA,37.619806,-122.374821,Overnight,53,F,10 mph,W,Mostly Cloudy,"Mostly cloudy, with a low around 53. West wind..."
3,KLAX,KLAX,LAX,Los Angeles International Airport,Los Angeles,US-CA,33.942501,-118.407997,Overnight,62,F,0 to 5 mph,S,Patchy Fog,"Patchy fog. Cloudy, with a low around 62. Sout..."
4,KOAK,KOAK,OAK,San Francisco Bay Oakland International Airport,Oakland,US-CA,37.720085,-122.221184,Today,74,F,7 to 16 mph,WSW,Mostly Sunny,"Mostly sunny. High near 74, with temperatures ..."


In [8]:
bq.load_table_from_dataframe(
    forecast_df,
    f"{PROJECT_ID}.{DATASET_ID}.{FORECAST_TBL}",
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
).result()
print(f"Wrote {len(forecast_df)} rows to `{FORECAST_TBL}`.")

Wrote 71 rows to `airport_forecasts`.


In [9]:
!bq mk --connection --location={REGION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE {CONNECTION_ID} 2>/dev/null || echo "Connection exists."

bq.query(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{REGION}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = 'gemini-2.5-flash')
""").result()
print("Gemini remote model ready.")

BigQuery error in mk operation: Already Exists: Connection
projects/931555514205/locations/us/connections/gemini_conn
Connection exists.
Gemini remote model ready.


In [10]:
bq.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TBL}` AS
SELECT
    * EXCEPT(prompt, ml_generate_text_rai_result, ml_generate_text_status),
    ml_generate_text_llm_result AS alert,
    CURRENT_TIMESTAMP() AS generated_at
FROM ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`,
    (
        SELECT
            *,
            CONCAT(
                'You are an FAA Aero Alerts officer. Write a brief, plain-language ',
                'weather advisory for travelers and airport staff at this airport. ',
                'Be clear about any hazards (wind, storms, snow, fog). Under 70 words. ',
                'Airport: ', name, ' (', IFNULL(iata_code, ident), '), ',
                municipality, ', ', iso_region,
                '. Forecast period: ', period_name,
                '. Temperature: ', CAST(temperature AS STRING), ' ', temperature_unit,
                '. Wind: ', wind_speed, ' from ', wind_direction,
                '. Conditions: ', detailed_forecast
            ) AS prompt
        FROM `{PROJECT_ID}.{DATASET_ID}.{FORECAST_TBL}`
    ),
    STRUCT(
        0.4  AS temperature,
        1024 AS max_output_tokens,
        TRUE AS flatten_json_output
    )
)
""").result()

bq.query(f"""
    SELECT name, iata_code, short_forecast, alert
    FROM `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TBL}`
    LIMIT 10
""").to_dataframe()

,name,iata_code,short_forecast,alert
0,Hartsfield Jackson Atlanta International Airport,ATL,Sunny,**ATL Weather Advisory: Today**\n\nExpect sunn...
1,Daniel K Inouye International Airport,HNL,Scattered Rain Showers,**HNL Overnight Weather Advisory:** Expect sca...
2,Kahului International Airport,OGG,Mostly Clear,**Aero Alert: Kahului (OGG) Overnight**\n\nOve...
3,Louisville Muhammad Ali International Airport,SDF,Sunny,**Aero Alert: SDF - Today**\n\nSunny skies and...
4,Cincinnati Northern Kentucky International Air...,CVG,Sunny,**Aero Alert: CVG - Today**\n\nSunny skies and...
5,Louis Armstrong New Orleans International Airport,MSY,Chance Showers And Thunderstorms then Showers ...,**FAA Aero Alert - MSY - Today**\n\nExpect sho...
6,Detroit Metropolitan Wayne County Airport,DTW,Sunny,**Aero Alert for DTW - Today:**\n\nSunny skies...
7,John F Kennedy International Airport,JFK,Sunny,**JFK Weather Advisory - Today:**\n\nExpect su...
8,George Bush Intercontinental Houston Airport,IAH,Chance Showers And Thunderstorms,**Aero Alert - IAH - Today**\n\nExpect mostly ...
9,Dallas Fort Worth International Airport,DFW,Partly Sunny then Slight Chance Showers And Th...,"DFW Advisory: Today, expect partly sunny skies..."


In [11]:
def run_pipeline():
    rows = []
    for _, a in large_df.iterrows():
        f = get_forecast(a["latitude_deg"], a["longitude_deg"])
        if f:
            rows.append({
                "ident": a["ident"], "icao_code": a["icao_code"],
                "iata_code": a["iata_code"], "name": a["name"],
                "municipality": a["municipality"], "iso_region": a["iso_region"],
                "latitude_deg": a["latitude_deg"], "longitude_deg": a["longitude_deg"],
                **f,
            })
        time.sleep(0.5)
    fdf = pd.DataFrame(rows)
    bq.load_table_from_dataframe(
        fdf, f"{PROJECT_ID}.{DATASET_ID}.{FORECAST_TBL}",
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE),
    ).result()

    bq.query(f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TBL}` AS
    SELECT * EXCEPT(prompt, ml_generate_text_rai_result, ml_generate_text_status),
           ml_generate_text_llm_result AS alert,
           CURRENT_TIMESTAMP() AS generated_at
    FROM ML.GENERATE_TEXT(
        MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`,
        (SELECT *, CONCAT(
            'You are an FAA Aero Alerts officer. Write a brief, plain-language ',
            'weather advisory for travelers and airport staff at this airport. ',
            'Be clear about any hazards (wind, storms, snow, fog). Under 70 words. ',
            'Airport: ', name, ' (', IFNULL(iata_code, ident), '), ',
            municipality, ', ', iso_region, '. Forecast period: ', period_name,
            '. Temperature: ', CAST(temperature AS STRING), ' ', temperature_unit,
            '. Wind: ', wind_speed, ' from ', wind_direction,
            '. Conditions: ', detailed_forecast) AS prompt
         FROM `{PROJECT_ID}.{DATASET_ID}.{FORECAST_TBL}`),
        STRUCT(0.4 AS temperature, 1024 AS max_output_tokens, TRUE AS flatten_json_output))
    """).result()
    print("Pipeline run complete.")

run_pipeline()

Pipeline run complete.
